# 🌐 LinguaBridge — NLLB-200 Fine-Tuning with LoRA

This notebook fine-tunes [NLLB-200-distilled-600M](https://huggingface.co/facebook/nllb-200-distilled-600M) using LoRA/PEFT for improved Indian language translation.

**Targets:** English ↔ Hindi, Punjabi, Tamil, Telugu, Bengali, Marathi, Urdu

**Dataset:** FLORES-200 dev/devtest splits

**Requirements:** GPU runtime (T4 or better recommended)

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft accelerate sentencepiece sacrebleu evaluate torch

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import evaluate
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if device == "cuda" else "N/A"}')

In [ ]:
# Configuration
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
LANG_PAIRS = [
    ('eng_Latn', 'hin_Deva'),   # English -> Hindi
    ('eng_Latn', 'pan_Guru'),   # English -> Punjabi
    ('eng_Latn', 'tam_Taml'),   # English -> Tamil
    ('eng_Latn', 'tel_Telu'),   # English -> Telugu
    ('eng_Latn', 'ben_Beng'),   # English -> Bengali
]
MAX_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
LORA_R = 8
LORA_ALPHA = 16

In [ ]:
# Load model and tokenizer
print('Loading NLLB-200 model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f'Model parameters: {model.num_parameters():,}')

In [ ]:
# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj'],
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load FLORES-200 dataset
print('Loading FLORES-200...')
flores = load_dataset('facebook/flores', 'all', trust_remote_code=True)
print(f'Train: {len(flores["dev"])} examples, Eval: {len(flores["devtest"])} examples')

In [ ]:
# Prepare dataset for a specific language pair
def prepare_data(dataset, src_lang, tgt_lang):
    """Create parallel corpus from FLORES-200."""
    src_col = f'sentence_{src_lang}'
    tgt_col = f'sentence_{tgt_lang}'
    
    # Check if columns exist
    available = [c for c in dataset.column_names if c.startswith('sentence_')]
    if src_col not in available or tgt_col not in available:
        # Try without script suffix
        src_col = f'sentence_{src_lang.split("_")[0]}'
        tgt_col = f'sentence_{tgt_lang.split("_")[0]}'
    
    def tokenize(examples):
        tokenizer.src_lang = src_lang
        model_inputs = tokenizer(
            examples[src_col],
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
        )
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                examples[tgt_col],
                max_length=MAX_LENGTH,
                truncation=True,
                padding='max_length',
            )
        model_inputs['labels'] = labels['input_ids']
        return model_inputs
    
    return dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)


# Prepare English -> Hindi data
src, tgt = LANG_PAIRS[0]
print(f'Preparing {src} -> {tgt}...')
train_ds = prepare_data(flores['dev'], src, tgt)
eval_ds = prepare_data(flores['devtest'], src, tgt)
print(f'Train: {len(train_ds)}, Eval: {len(eval_ds)}')

In [ ]:
# Metrics
bleu_metric = evaluate.load('sacrebleu')
chrf_metric = evaluate.load('chrf')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # Decode
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # BLEU
    bleu = bleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    chrf = chrf_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {
        'bleu': round(bleu['score'], 2),
        'chrf': round(chrf['score'], 2),
    }

In [ ]:
# Training
training_args = Seq2SeqTrainingArguments(
    output_dir='./lora_nllb_output',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()

In [ ]:
# Save LoRA adapter
model.save_pretrained('./lora_nllb_adapter')
print('LoRA adapter saved to ./lora_nllb_adapter')

# Evaluate final model
results = trainer.evaluate()
print(f'\n📊 Final Results:')
print(f'   BLEU:  {results.get("eval_bleu", "N/A")}')
print(f'   chrF:  {results.get("eval_chrf", "N/A")}')

In [ ]:
# Quick test
test_sentences = [
    'Hello, how are you?',
    'The weather is beautiful today.',
    'Machine learning is transforming the world.',
    'Can you help me find the nearest hospital?',
]

print('\n🔤 Translation Test (English → Hindi):')
print('=' * 60)
tokenizer.src_lang = 'eng_Latn'
for sent in test_sentences:
    inputs = tokenizer(sent, return_tensors='pt', max_length=128, truncation=True)
    tgt_id = tokenizer.convert_tokens_to_ids('hin_Deva')
    with torch.no_grad():
        output = model.generate(**inputs, forced_bos_token_id=tgt_id, max_new_tokens=128)
    translation = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f'  EN: {sent}')
    print(f'  HI: {translation}')
    print()